# Customer 360 View
create customer 360 view with columns
- customer_id
- customer_name
- home_branch_name
- total_accounts
- total_balance
- total_transactions
- total_transaction_amount
- credit_score
- risk_grade
- external_active_loans
- external_overdue_amount
- customer_segment \
balance \
(0 to 1 lakh - low value) \
(1 lakh to 5 lakhs - medium value) \
(more than 5 lakhs) - high value

In [0]:
CREATE OR REPLACE VIEW neo_bank.gold.vw_customer_360 AS

WITH customer_agg AS (
        select 
        customer_sk, 
        customer_id,
        concat(dc.first_name, ' ', dc.last_name) as customer_name,
        branch_name as home_branch_name
    from neo_bank.gold.dim_customers dc
    inner join neo_bank.gold.dim_branches db
        on dc.home_branch_sk = db.branch_sk
    group by customer_sk, customer_id, customer_name,home_branch_name
    order by customer_sk
),

accounts_agg AS (
    select 
        count(account_id) as total_accounts,
        sum(balance) as total_balance,customer_sk 
    from neo_bank.gold.dim_accounts group by customer_sk order by customer_sk
),

transaction_agg AS (
    select count(txn_id) as total_transactions, sum(amount) as total_transaction_amount, customer_sk from neo_bank.gold.fact_transactions group by customer_sk order by customer_sk
),

-- Get only the most recent credit bureau report per customer
credit_bureau_latest AS (
    SELECT 
        customer_sk,
        credit_score,
        risk_grade,
        external_active_loans,
        external_overdue_amount,
        ROW_NUMBER() OVER (PARTITION BY customer_sk ORDER BY bureau_pull_date_key DESC) as rn
    FROM neo_bank.gold.fact_credit_bureau_reports
    QUALIFY rn = 1
)

select 
    c.customer_id,
    c.customer_name,
    c.home_branch_name,
    coalesce(a.total_accounts,0) as total_accounts,
    coalesce(a.total_balance,0) as total_balance,
    coalesce(t.total_transactions,0) as total_transactions,
    coalesce(t.total_transaction_amount,0) as total_transaction_amount,
    cbr.credit_score,
    cbr.risk_grade,
    cbr.external_active_loans,
    cbr.external_overdue_amount,
    CASE 
        WHEN total_balance < 100000 THEN 'LOW_VALUE'
        WHEN total_balance >100000 AND total_balance<500000 THEN 'MEDIUM_VALUE'
        WHEN total_balance > 500000 THEN 'HIGH_VALUE'
        ELSE 'LOW_VALUE'
    END AS customer_segment

from customer_agg c
left join accounts_agg a on c.customer_sk ==a.customer_sk
left join transaction_agg t on c.customer_sk == t.customer_sk
left join credit_bureau_latest cbr on c.customer_sk == cbr.customer_sk
